<a href="https://colab.research.google.com/github/jdasam/aat3020/blob/2025/notebooks/1_word2vec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Word2Vec Implementation from Scratch

This notebook demonstrates how to implement the Word2Vec algorithm from scratch using PyTorch. We'll use the first Harry Potter book as our corpus to train word embeddings.


## 1. Setting Up the Environment

First, we need to import the necessary libraries:
- `torch` and `torch.nn` for tensor operations and neural network functionality
- `string` for string manipulations (removing punctuation)


In [24]:
import torch as th
import torch.nn as nn
import string


## 2. Getting the Text Data

We'll download the first Harry Potter book to use as our corpus.

In [7]:
!wget "https://raw.githubusercontent.com/amephraim/nlp/master/texts/J.%20K.%20Rowling%20-%20Harry%20Potter%201%20-%20Sorcerer's%20Stone.txt"


--2025-03-18 13:38:04--  https://raw.githubusercontent.com/amephraim/nlp/master/texts/J.%20K.%20Rowling%20-%20Harry%20Potter%201%20-%20Sorcerer's%20Stone.txt
raw.githubusercontent.com (raw.githubusercontent.com) 해석 중... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
다음으로 연결 중: raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... 연결했습니다.
HTTP 요청을 보냈습니다. 응답 기다리는 중... 200 OK
길이: 439742 (429K) [text/plain]
저장 위치: `J. K. Rowling - Harry Potter 1 - Sorcerer\'s Stone.txt.2'

J. K. Rowling - Har 100%[===================>] 429.44K  2.26MB/s    /  0.2s    

2025-03-18 13:38:04 (2.26 MB/s) - `J. K. Rowling - Harry Potter 1 - Sorcerer\'s Stone.txt.2' 저장함 [439742/439742]



## 3. Text Preprocessing

Before we can use the text data, we need to preprocess it:
- Remove punctuation
- Convert text to lowercase
- Split text into tokens (words)

This function will help us clean and tokenize the text.

In [8]:
def remove_punctuation(x):
  return x.translate(''.maketrans('', '', string.punctuation))

def make_tokenized_corpus(corpus):
  out= [ [y.lower() for y in remove_punctuation(sentence).split(' ') if y] for sentence in corpus]
  return [x for x in out if x!=[]]


## 4. Loading and Formatting the Text

Now we'll load the text file, replace some special characters, and split the text into sentences.


In [9]:
with open("J. K. Rowling - Harry Potter 1 - Sorcerer's Stone.txt", 'r') as f:
  strings = f.readlines()
sample_text = "".join(strings).replace('\n', ' ').replace('Mr.', 'mr').replace('Mrs.', 'mrs').split('. ')


Let's tokenize the text using our preprocessing function `make_tokenized_corpus`:

In [4]:
# Corpus is a list of list of strings (words)

In [10]:
corpus = make_tokenized_corpus(sample_text)

corpus[:5]

[['harry',
  'potter',
  'and',
  'the',
  'sorcerers',
  'stone',
  'chapter',
  'one',
  'the',
  'boy',
  'who',
  'lived',
  'mr',
  'and',
  'mrs',
  'dursley',
  'of',
  'number',
  'four',
  'privet',
  'drive',
  'were',
  'proud',
  'to',
  'say',
  'that',
  'they',
  'were',
  'perfectly',
  'normal',
  'thank',
  'you',
  'very',
  'much'],
 ['they',
  'were',
  'the',
  'last',
  'people',
  'youd',
  'expect',
  'to',
  'be',
  'involved',
  'in',
  'anything',
  'strange',
  'or',
  'mysterious',
  'because',
  'they',
  'just',
  'didnt',
  'hold',
  'with',
  'such',
  'nonsense'],
 ['mr',
  'dursley',
  'was',
  'the',
  'director',
  'of',
  'a',
  'firm',
  'called',
  'grunnings',
  'which',
  'made',
  'drills'],
 ['he',
  'was',
  'a',
  'big',
  'beefy',
  'man',
  'with',
  'hardly',
  'any',
  'neck',
  'although',
  'he',
  'did',
  'have',
  'a',
  'very',
  'large',
  'mustache'],
 ['mrs',
  'dursley',
  'was',
  'thin',
  'and',
  'blonde',
  'and',
  'had

## 5. Creating Context Word Pairs

A key concept in Word2Vec is learning from context. We need to create pairs of words that appear near each other in the text. We'll use a sliding window approach to create these pairs.

For example, with the window size of 2, for the word "to" in the sentence "they were the last people youd expect to be involved...", we would create pairs with:
- ("to", "expect")
- ("to", "be")
- ("to", "involved")
- ("to", "in")

These pairs will be our training data.

In [11]:
from tqdm import tqdm

sample_sentence = ['they', 'were', 'the', 'last', 'people', 'youd', 'expect', 'to', 'be', 'involved', 'in', 'anything', 'strange', 'or', 'mysterious', 'because', 'they', 'just', 'didnt', 'hold', 'with', 'such', 'nonsense']

word_pairs = []
window_size = 2

for sample_sentence in tqdm(corpus):
  for cur_idx, center_word in enumerate(sample_sentence):
    window_begin = max(cur_idx - window_size, 0)
    window_end = min(cur_idx + window_size + 1, len(sample_sentence))
    # for context_word in sample_sentence[window_begin:window_end]:
    #   # if center_word == context_word: continue
    #   word_pairs.append( (center_word, context_word))
    for j in range(window_begin, window_end):
      if cur_idx == j: continue
      word_pairs.append( (center_word, sample_sentence[j]))

print(f"\nLength of word_pairs is {len(word_pairs)}")
print(f"First 5 example of word_pairs is {word_pairs[:5]}")

100%|██████████| 4682/4682 [00:00<00:00, 48122.73it/s]


Length of word_pairs is 282372
First 5 example of word_pairs is [('harry', 'potter'), ('harry', 'and'), ('potter', 'harry'), ('potter', 'and'), ('potter', 'the')]


## 6. Building the Vocabulary

To work with word vectors, we need to create a vocabulary that maps each unique word to an index. We'll also filter out rare words that appear less than a certain number of times in the corpus.

### 6.1 Collecting All Words

First, let's collect all words in our corpus:


In [12]:
# we have to make vocabulary

entire_words = []

for sentence in tqdm(corpus):
  for word in sentence:
    entire_words.append(word)

len(entire_words)

100%|██████████| 4682/4682 [00:00<00:00, 519241.97it/s]


77597


### 6.2 Finding Unique Words

Now, let's find the unique words in our corpus:

In [13]:
# we have to get the "unique" item among total words
unique_words = set(entire_words)
len(unique_words)

6038

### 6.3 Converting to a List and Sorting

We'll convert the set of unique words to a sorted list:

In [14]:
# vocab_set[0] # set is not subscriptable because it has no order
unique_words = list(unique_words)
unique_words.sort()
unique_words[:5]

['\the', '0', '1', '1473', '1637']

### 6.4 Filtering by Frequency

Now, let's filter out rare words that occur less than a specified number of times:
- We can use the `Counter` class from the `collections` module to count the frequency of each word in the corpus.
- Caution on `alist.sort()` will return `None`.

In [15]:
# how can we filter the vocab by its frequency?
filtered_vocab = None
# you can use word counter as dictionary
# In python dictionary, dict.keys() gives keys, and dict.values() give values,
# dict.items() give (key, value)

from collections import Counter
word_counter = Counter(entire_words)
word_counter
word_counter.most_common(10)
word_counter['ron'] # word_counter.get('ron')

threadhold = 5
filtered_vocab = []

for key, value in word_counter.items():
  if value > threadhold:
    filtered_vocab.append(key)

filtered_vocab.sort()
len(filtered_vocab)

1506

## 7. Filtering Word Pairs

Now that we have our filtered vocabulary, we need to filter our word pairs to only include words that are in our vocabulary:

In [16]:
# Filter the word_pairs using the vocab
# word_pairs, filtered_vocab
# word_pairs is a list of [word_a, word_b]

filtered_pairs =[]
vocab_set = set(filtered_vocab) # for increasing the speed
for word_a, word_b in tqdm(word_pairs):
  if word_a in vocab_set and word_b in vocab_set:
    filtered_pairs.append((word_a, word_b))

100%|██████████| 282372/282372 [00:00<00:00, 2626667.81it/s]


In [17]:
# implement same algorithm with list comprehension

filtered_pairs = [(word_a, word_b) for word_a, word_b in tqdm(word_pairs)
                  if word_a in vocab_set and word_b in vocab_set]

100%|██████████| 282372/282372 [00:00<00:00, 3262692.21it/s]


In [18]:
len(filtered_pairs), len(word_pairs)

(226846, 282372)

## 8. Converting Words to Indices

For efficiency, we'll convert our words to indices according to their position in our vocabulary:

In [19]:
# convert word into index of vocab
filtered_vocab.index('harry')


527

This is inefficient because `list.index()` has to scan the list every time. Let's use a dictionary for faster lookups:

In [22]:
# we can make it faster
# use dictionary to find the index of string

word2idx = dict()
for idx, word in enumerate(filtered_vocab):
    word2idx[word] = idx

word2idx['harry']

527

Now, let's convert our word pairs to index pairs more efficiently:

In [ ]:
index_pairs = []

# for word_a, word_b in tqdm(filtered_pairs):
#     index_pairs.append((word2idx[word_a], word2idx[word_b]))

index_pairs = [(word2idx[word_a], word2idx[word_b]) for word_a, word_b in tqdm(filtered_pairs)]

100%|██████████| 226846/226846 [00:00<00:00, 3257223.65it/s]


(527, 953)

In [28]:
# Why we don't need idx2tok?
filtered_vocab[527]

'harry'

## 9. Creating Initial Word Vectors

Now we'll create random vectors for each word in our vocabulary. These vectors will be adjusted during training:
- We can use `torch.randn` to create random vectors that follow normal distribution.

In [ ]:
# we have to make random vectors for each word in the vocab
# we also have to decide the dimension of the vector

dim = 100
vocab_size = len(filtered_vocab)

word_vectors = th.randn(vocab_size, dim) / 10


tensor(0.7869)
tensor(-1.4037)


In [41]:
# what is the vector for harry?
word_vectors[word2idx['harry']]

tensor([    -0.1043,     -0.1168,     -0.0221,      0.0004,      0.0455,
            -0.0531,      0.0029,      0.0315,     -0.0466,     -0.1137,
             0.0382,     -0.0362,      0.0681,     -0.0190,     -0.0647,
             0.0078,     -0.0001,      0.0552,     -0.0596,     -0.0331,
            -0.0961,     -0.0631,      0.1342,      0.2182,     -0.1874,
            -0.0136,      0.0688,     -0.0926,      0.0082,      0.0520,
             0.1481,     -0.1246,     -0.1459,      0.0811,     -0.0101,
             0.0088,      0.0228,     -0.0356,      0.0917,     -0.0477,
            -0.0424,     -0.0034,     -0.0372,      0.0674,      0.1204,
             0.0377,      0.0702,     -0.0157,      0.1058,      0.0991,
             0.0611,      0.1337,      0.1657,      0.0404,     -0.0625,
             0.0439,     -0.1714,      0.0192,      0.0261,      0.1232,
            -0.1555,     -0.0981,     -0.0528,      0.0039,     -0.2211,
             0.0523,      0.0795,     -0.1841,     

## 10. Understanding Word Relationships with Dot Products

The core of Word2Vec is using dot products to measure relationships between words. Let's explore this concept:

In [42]:
th.set_printoptions(sci_mode=False) # Do this to avoid scientific notation


## Dot Product
- Assume we have two vectors $a$ and $b$.
  - $a = [a_1, a_2, a_3, a_4, ..., a_n]$
  - $b = [b_1, b_2, b_3, b_4, ..., b_n]$
- $a \cdot b$ = $\sum _{i=1}^n a_ib_i$  = $a_1b_1 + a_2b_2 + a_3b_3 + a_4b_4 + ... + a_nb_n$

Let's calculate the dot product between "harry" and "potter":


In [ ]:
# calculate P(potter|harry)
dot_product_value_between_potter_harry = th.dot(word_vectors[word2idx['harry']], word_vectors[word2idx['potter']])
# dot_product_value_between_potter_harry = sum(word_vectors[word2idx['harry']]* word_vectors[word2idx['potter']])
dot_product_value_between_potter_harry

tensor(-0.1669)

In [61]:
# we can get the dot product value for every other words in the vocab
# to get  P(word | harry)
word_dot_dict = {}

for word in tqdm(filtered_vocab):
  word_dot_dict[word] = th.dot(word_vectors[word2idx['harry']], word_vectors[word2idx[word]])
  # word_dot_dict[word] = sum(word_vectors[word2idx['harry']] * word_vectors[word2idx[word]])

word_dot_dict

100%|██████████| 1506/1506 [00:00<00:00, 191042.28it/s]


{'a': tensor(0.1069),
 'able': tensor(0.0025),
 'abou': tensor(-0.0757),
 'about': tensor(0.0994),
 'above': tensor(-0.1268),
 'across': tensor(-0.1383),
 'added': tensor(-0.0266),
 'afford': tensor(0.0192),
 'afraid': tensor(0.0195),
 'after': tensor(-0.0467),
 'afternoon': tensor(0.0441),
 'again': tensor(-0.0065),
 'against': tensor(0.1334),
 'ages': tensor(-0.0392),
 'ago': tensor(-0.0108),
 'agreed': tensor(0.1484),
 'ah': tensor(0.0339),
 'ahead': tensor(0.0977),
 'air': tensor(-0.1575),
 'albus': tensor(-0.1222),
 'alive': tensor(-0.0397),
 'all': tensor(-0.0704),
 'alley': tensor(0.0651),
 'allowed': tensor(-0.0789),
 'almost': tensor(-0.0197),
 'alone': tensor(-0.0731),
 'along': tensor(-0.1284),
 'already': tensor(0.0203),
 'also': tensor(-0.0867),
 'although': tensor(0.0167),
 'always': tensor(-0.2172),
 'am': tensor(0.0014),
 'an': tensor(-0.0120),
 'and': tensor(0.0428),
 'angrily': tensor(-0.0968),
 'angry': tensor(-0.0050),
 'another': tensor(0.0214),
 'answer': tensor(-

Now, let's convert these dot products to probabilities using the softmax function:
- We have to convert our prediction into probability distribution to get P(word|harry) so that sum of [P(a|harry), ..., P(potter|harry), ... P(ron|harry), ... ] = 1
- current dot product value is any real number, sometimes called as logit
  - logit from logistic regression. Some values that are not yet converted to 0-1 or value before sigmoid function
  - every probability should be in range (0, 1) (greater than 0, smaller than 1)
  - this can be handled by taking exponential of dot product values, divided by total sum
  - This function is called **Softmax**

- Why we use exponential?
  - Because we want to make every probability in positive range while preserving the order


In [ ]:
from math import exp

word_exp_dict = {}
for word, dot_value in tqdm(word_dot_dict.items()):
  word_exp_dict[word] = exp(dot_value)

word_prob_dict = {}
for word, exp_value in tqdm(word_exp_dict.items()):
  word_prob_dict[word] = exp_value / sum(word_exp_dict.values())

word_prob_dict

100%|██████████| 1506/1506 [00:00<00:00, 107272.30it/s]


In [ ]:
# Get P(potter|harry)
# word_exp_dict['potter'] / sum(word_exp_dict.values())
word_prob_dict['potter']

0.0006936211360456018

## 13. Efficient Matrix Operations
![img](https://mkang32.github.io/images/python/khan_academy_matrix_product.png)

Instead of calculating dot products one by one, we can use matrix multiplication for efficiency:


In [ ]:
# get dot product result for every word in the vocabulary

# first, make vector_of_harry into matrix format

# do matrix multiplication


Let's verify that our matrix multiplication gives the same result as individual dot products:

Now let's implement the complete softmax calculation using matrix operations:


In [ ]:
# convert dot product result into exponential

In [ ]:
# get the sum of exponential


In [ ]:
# divide exponential value with sum

## 14. Creating a Probability Function

Let's create a function to calculate probabilities efficiently:

In [ ]:
def get_probs(query_vectors, entire_vectors):
  return None

# get_probs(mat_of_harry, word_vectors)

## 15. Preparing for Training

Before training our Word2Vec model, we need to split our dataset into training and testing sets:

In [ ]:
# Now we can train the word2vec

# Let's think about training pairs
index_pairs # this is our dataset. It's list of list of two integer
# two integer means a pair of neighboring words

# Training set and Test set
# To validate that our model can solve 'unseen' problems
# So we have to split the dataset before training.

# To randomly split the dataset, we will first shuffle the dataset

# random.shuffle(index_pairs) # this will shuffle the list items

In [ ]:
len(train_set), len(test_set)

## 16. Training the Word2Vec Model

Now we'll train our Word2Vec model using batched gradient descent:

In [ ]:
# making batch from train_set
# Batch is a set of training samples, that are calculated together
# And also we update the model after one single batch

## 17. Evaluating the Training

Let's visualize the training loss to see if our model is learning:

In [ ]:
import matplotlib.pyplot as plt
plt.plot(loss_record)

## 18. Testing the Model

Now we'll test our model on the test set:

## 19. Exploring Learned Word Relationships

Let's explore what our model has learned by finding the words most closely related to "harry":

In [ ]:
# P(potter|harry)?
